# Test Predictions Export and Firm-Clustered Bootstrap (v3)

Owner: Jessica Bedson. Follow-up to Tasks 1.5 and 1.6 for the evaluation harness (Mouzam, `evaluate.py`).

What this notebook does:

1. Rebuilds the v3 test set exactly as `Tasks 1.1-1.6 Model v3.ipynb` does (label filter, t+1 pairs, features, Enoch's test window).
2. Scores it with the **frozen** artifacts in `models/`. Nothing is refitted, so the frozen models and every verified number stay as they are.
3. Checks that the pooled test ROC-AUC of both models matches `results/test_metrics_v3.csv`, and stops if it does not.
4. Writes `results/test_predictions_v3.csv`: `gvkey, feature_year, outcome_year, y, p_financial, p_prior, current_adverse`.
5. Writes firm-clustered bootstrap intervals, as the evaluation design specifies: `results/test_bootstrap_ci_firm_v3.csv` and `results/test_model_gap_firm_v3.csv`. The notebook's `test_bootstrap_ci_v3.csv` resampled firm-year rows; it is kept unchanged for the record.

Run it in the folder that holds `final_panel 1.csv`, `models/` and `results/`



## Setup

In [1]:
from __future__ import annotations

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score

DECISION_LOG_SEED = 20260928
try:
    import config
except ImportError:
    config = None
RANDOM_SEED = int(getattr(config, "SEED", DECISION_LOG_SEED))
assert RANDOM_SEED == DECISION_LOG_SEED, "Seed differs from the decision log."

DATA_PATH = Path(getattr(config, "DATA_PATH", "final_panel 1.csv"))
MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")
N_BOOTSTRAP = 1000

FEATURE_COLS = [
    "LNEMP", "LNTA", "LNSALE", "LNTI", "IntanTA", "InvCA", "CapInt",
    "TobinQ", "RET1Y", "ROA", "ROE", "GPM", "CURR_RATIO", "AT_TURN",
    "INV_TURN", "DPR", "PTB", "PE", "cfsale",
]
PRIOR_COL = "prior_adverse_opinion"
TEST_YEARS = (2018, 2021)
TEST_OUTCOME_YEARS = [2019, 2020, 2021, 2022]

## Rules copied from the v3 notebook

Same label filter, t+1 pairing, feature rule and test window as Tasks 1.1 to 1.4.

In [2]:
# --- same rules as the notebook ------------------------------------------------

def prepare_features(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    sales = pd.to_numeric(out["sale"], errors="raise")
    income = pd.to_numeric(out["oibdp"], errors="raise")
    out["cfsale"] = income / sales.replace(0, np.nan)
    out[FEATURE_COLS] = (
        out[FEATURE_COLS].apply(pd.to_numeric, errors="raise").replace([np.inf, -np.inf], np.nan)
    )
    return out


def log_odds(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


def pr_auc(y, p):
    precision, recall, _ = precision_recall_curve(y, p)
    return auc(recall, precision)


def build_test_set(raw: pd.DataFrame) -> pd.DataFrame:
    assessed = raw.loc[raw["auopic"].isin([1, 2])].copy()
    assessed["target"] = (assessed["auopic"] == 2).astype(int)
    assessed["feature_year"] = assessed["fyear"].astype(int)
    assessed["prediction_date"] = assessed["INFO_DATE"]
    assessed[PRIOR_COL] = assessed["target"]

    outcomes = assessed[["gvkey", "feature_year", "INFO_DATE", "target"]].rename(columns={
        "feature_year": "outcome_year", "INFO_DATE": "outcome_info_date", "target": "outcome_target",
    })
    pairs = assessed.copy()
    pairs["outcome_year"] = pairs["feature_year"] + 1
    pairs = pairs.merge(outcomes, on=["gvkey", "outcome_year"], how="inner", validate="one_to_one")
    pairs["target"] = pairs["outcome_target"].astype(int)
    pairs = pairs.sort_values(["prediction_date", "gvkey"]).reset_index(drop=True)
    pairs = prepare_features(pairs)

    test = pairs.loc[pairs["feature_year"].between(*TEST_YEARS)].copy()
    return test.sort_values(["prediction_date", "gvkey"]).reset_index(drop=True)


def calibrated_scores(artifact: dict, frame: pd.DataFrame) -> np.ndarray:
    raw_p = artifact["pipeline"].predict_proba(frame[artifact["feature_cols"]])[:, 1]
    return artifact["calibrator"].predict_proba(log_odds(raw_p))[:, 1]

## Firm-clustered bootstrap

Each draw resamples **firms** with replacement and keeps all of a firm's test rows together. Within a single outcome year each firm has one row, so per-year intervals are unaffected by the choice; the pooled 2019-2022 interval is where clustering matters, because firms repeat across years.

In [3]:
# --- firm-clustered bootstrap --------------------------------------------------

def firm_resamples(firms: np.ndarray, n: int, seed: int):
    """Yield row indices for n bootstrap samples that resample firms with replacement."""
    rng = np.random.default_rng(seed)
    codes, uniques = pd.factorize(firms)
    rows_by_firm = [np.flatnonzero(codes == k) for k in range(len(uniques))]
    for _ in range(n):
        picked = rng.integers(0, len(uniques), len(uniques))
        yield np.concatenate([rows_by_firm[k] for k in picked])


def firm_bootstrap(y, firms, scores: dict, n=N_BOOTSTRAP, seed=RANDOM_SEED):
    """ROC/PR intervals for each model and for the gap (prior minus financial) on the same resamples."""
    y = np.asarray(y)
    draws = {name: {"roc": [], "pr": []} for name in scores}
    gap = {"roc": [], "pr": []}
    for idx in firm_resamples(np.asarray(firms), n, seed):
        yb = y[idx]
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        roc = {k: roc_auc_score(yb, s[idx]) for k, s in scores.items()}
        pra = {k: pr_auc(yb, s[idx]) for k, s in scores.items()}
        for k in scores:
            draws[k]["roc"].append(roc[k])
            draws[k]["pr"].append(pra[k])
        gap["roc"].append(roc["prior"] - roc["financial"])
        gap["pr"].append(pra["prior"] - pra["financial"])
    pct = lambda v, q: float(np.percentile(v, q))
    ci = {k: {"ROC_lo": pct(d["roc"], 2.5), "ROC_hi": pct(d["roc"], 97.5),
              "PR_lo": pct(d["pr"], 2.5), "PR_hi": pct(d["pr"], 97.5),
              "draws_used": len(d["roc"])} for k, d in draws.items()}
    g = {"ROC_gap": roc_auc_score(y, scores["prior"]) - roc_auc_score(y, scores["financial"]),
         "ROC_lo": pct(gap["roc"], 2.5), "ROC_hi": pct(gap["roc"], 97.5),
         "PR_gap": pr_auc(y, scores["prior"]) - pr_auc(y, scores["financial"]),
         "PR_lo": pct(gap["pr"], 2.5), "PR_hi": pct(gap["pr"], 97.5)}
    return ci, g

## Load the frozen models and rebuild the test set

In [4]:
decisions = json.loads((MODEL_DIR / "decisions_v3.json").read_text(encoding="utf-8"))
primary_name = decisions["selection"]["primary"]
variant_name = decisions["selection"]["variant"]
arts = {
    "financial": pickle.loads((MODEL_DIR / f"{decisions['primary_version']}.pkl").read_bytes()),
    "prior": pickle.loads((MODEL_DIR / f"{decisions['variant_version']}.pkl").read_bytes()),
}

raw = pd.read_csv(DATA_PATH, dtype={"gvkey": "string", "cik": "string", "cusip": "string", "tic": "string"},
                  parse_dates=["datadate", "INFO_DATE"])
test = build_test_set(raw)
y = test["target"].to_numpy()
scores = {k: calibrated_scores(a, test) for k, a in arts.items()}

## Check against the reported results

Stops if the rebuilt test set does not reproduce `test_metrics_v3.csv` exactly.

In [5]:
# Stop if the rebuilt test set does not reproduce the reported pooled results.
reported = pd.read_csv(RESULTS_DIR / "test_metrics_v3.csv")
pooled_label = f"Pooled {TEST_OUTCOME_YEARS[0]}-{TEST_OUTCOME_YEARS[-1]}"
for key, name in [("financial", primary_name), ("prior", variant_name)]:
    row = reported[(reported.model == f"{name} (calibrated)") & (reported.subgroup == pooled_label)].iloc[0]
    assert int(row.rows) == len(test), f"Test rows {len(test)} differ from reported {int(row.rows)}."
    got = roc_auc_score(y, scores[key])
    assert abs(got - row.ROC_AUC) < 1e-9, f"{name}: ROC-AUC {got} differs from reported {row.ROC_AUC}."
print(f"Rebuilt test set matches test_metrics_v3.csv: {len(test):,} rows, "
      f"ROC-AUC {roc_auc_score(y, scores['financial']):.4f} / {roc_auc_score(y, scores['prior']):.4f}")

Rebuilt test set matches test_metrics_v3.csv: 12,325 rows, ROC-AUC 0.6864 / 0.8064


## Export firm-year predictions

In [6]:
predictions = pd.DataFrame({
    "gvkey": test["gvkey"].astype(str),
    "feature_year": test["feature_year"].astype(int),
    "outcome_year": test["outcome_year"].astype(int),
    "y": y.astype(int),
    "p_financial": scores["financial"],
    "p_prior": scores["prior"],
    "current_adverse": test[PRIOR_COL].astype(int),
})
assert predictions[["p_financial", "p_prior"]].notna().all().all()
assert set(predictions["current_adverse"].unique()) <= {0, 1}
out = RESULTS_DIR / "test_predictions_v3.csv"
predictions.to_csv(out, index=False)
print(f"Wrote {out} ({len(predictions):,} firm-years, {predictions.gvkey.nunique():,} firms)")

Wrote results/test_predictions_v3.csv (12,325 firm-years, 3,785 firms)


## Firm-clustered intervals, pooled and by outcome year

In [7]:
# Firm-clustered bootstrap, pooled and by outcome year.
groups = [(pooled_label, np.ones(len(test), dtype=bool))]
groups += [(f"Outcome {yr}", test["outcome_year"].to_numpy() == yr) for yr in TEST_OUTCOME_YEARS]
ci_rows, gap_rows = [], []
for label, mask in groups:
    sub_scores = {k: s[mask] for k, s in scores.items()}
    ci, g = firm_bootstrap(y[mask], test["gvkey"].to_numpy()[mask], sub_scores)
    for key, name in [("financial", primary_name), ("prior", variant_name)]:
        ci_rows.append({"model": name, "subgroup": label, "resampling": "firms", **ci[key]})
    gap_rows.append({"subgroup": label, "resampling": "firms", **g})
ci_table, gap_table = pd.DataFrame(ci_rows), pd.DataFrame(gap_rows)
ci_table.to_csv(RESULTS_DIR / "test_bootstrap_ci_firm_v3.csv", index=False)
gap_table.to_csv(RESULTS_DIR / "test_model_gap_firm_v3.csv", index=False)
print("\nFirm-clustered 95% intervals (calibrated):")
print(ci_table.round(4).to_string(index=False))
print("\nGap, prior-opinion minus financial-only (firm-clustered):")
print(gap_table.round(4).to_string(index=False))


Firm-clustered 95% intervals (calibrated):
                  model         subgroup resampling  ROC_lo  ROC_hi  PR_lo  PR_hi  draws_used
                XGBoost Pooled 2019-2022      firms  0.6604  0.7153 0.0647 0.1182        1000
XGBoost + prior opinion Pooled 2019-2022      firms  0.7799  0.8314 0.2387 0.3475        1000
                XGBoost     Outcome 2019      firms  0.6377  0.7310 0.0677 0.1335        1000
XGBoost + prior opinion     Outcome 2019      firms  0.7464  0.8392 0.2037 0.3456        1000
                XGBoost     Outcome 2020      firms  0.6306  0.7376 0.0441 0.1101        1000
XGBoost + prior opinion     Outcome 2020      firms  0.7436  0.8597 0.1672 0.3403        1000
                XGBoost     Outcome 2021      firms  0.6006  0.7058 0.0357 0.0956        1000
XGBoost + prior opinion     Outcome 2021      firms  0.7781  0.8682 0.1547 0.3263        1000
                XGBoost     Outcome 2022      firms  0.6666  0.7449 0.0898 0.1619        1000
XGBoost + prior 